In [0]:
result = spark.sql("""
    SELECT ai_query(
        'databricks-meta-llama-3-3-70b-instruct',
        'Reply with just the word: working'
    ) AS response
""")
display(result)

In [0]:
from pyspark.sql import functions as F
import pandas as pd

spark.sql("USE loan_risk")
print(" Config loaded")
print(" Database: loan_risk")

In [0]:
# Read schema from Gold table
# This tells the LLM what columns are available
df = spark.table("loan_risk.gold_loans_scored")

col_defs = ", ".join(
    f"{f.name} ({f.dataType.simpleString()})"
    for f in df.schema.fields
)

row_count = df.count()

SCHEMA = f"""
TABLE: loan_risk.gold_loans_scored
ROWS : {row_count:,}
COLUMNS: {col_defs}
"""

print(" Schema loaded:")
print(SCHEMA)

In [0]:
SYSTEM_PROMPT = f"""
You are a SQL expert for a financial loan database.
Convert natural language questions into valid Spark SQL queries.

DATABASE SCHEMA:
{SCHEMA}

KEY COLUMN NOTES:
- default: 1 = defaulted, 0 = fully paid
- risk_score: 3-13, higher = riskier
- risk_label: Low, Medium, High, Very High
- default_probability: 0.0 to 1.0 (ML model output)
- grade: A (best credit) to G (worst credit)
- int_rate: interest rate in percent
- dti: debt-to-income ratio
- addr_state: 2-letter US state code (TX=Texas, CA=California etc)

RULES:
1. Return ONLY the SQL query — no explanation, no markdown, no backticks
2. Always use full table name: loan_risk.gold_loans_scored
3. For high risk use: risk_label IN ('High', 'Very High')
4. Limit results to 20 rows unless user specifies
5. Use Spark SQL syntax only
6. ALWAYS add this filter to every query:
   WHERE LENGTH(addr_state) = 2
   If query already has WHERE, add: AND LENGTH(addr_state) = 2

"""

print(" System prompt updated")

In [0]:
def question_to_sql(question: str) -> str:
    """Convert natural language question to SQL using Databricks LLM"""
    
    prompt = SYSTEM_PROMPT + f"User question: {question}\nSQL:"
    
    result = spark.sql(f"""
        SELECT ai_query(
            'databricks-meta-llama-3-3-70b-instruct',
            '{prompt.replace("'", "''")}' 
        ) AS sql_query
    """).collect()[0]["sql_query"]
    
    # Clean up any accidental markdown
    sql = result.strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()
    
    return sql

print(" question_to_sql function ready")

In [0]:
# Test with a simple question
question = "Show top 10 loans with highest default probability"
sql = question_to_sql(question)
print(f"Question: {question}")
print(f"\nGenerated SQL:\n{sql}")

In [0]:
def run_query(sql: str):
    """Execute SQL against Delta table and return results"""
    try:
        result = spark.sql(sql)
        return result
    except Exception as e:
        print(f"❌ Query failed: {e}")
        return None

# Test it
result = run_query(sql)
display(result)

In [0]:
def generate_insight(question: str, result_df) -> str:
    """Generate plain English insight from query results"""
    
    # Get summary of results
    pdf = result_df.limit(5).toPandas()
    summary = pdf.to_string(index=False)
    
    prompt = f"""
A financial analyst asked: '{question}'
Here are the top results:
{summary}

Give a 2 sentence plain English insight a bank manager would understand.
Be specific with numbers from the data.
"""
    
    insight = spark.sql(f"""
        SELECT ai_query(
            'databricks-meta-llama-3-3-70b-instruct',
            '{prompt.replace("'", "''")}'
        ) AS insight
    """).collect()[0]["insight"]
    
    return insight.strip()

# Test it
insight = generate_insight(question, result)
print(f" Insight: {insight}")

In [0]:
def ask(question: str):
    """
    Main chatbot function.
    Type any question in plain English and get SQL + results + insight.
    """
    print(f"\n{'='*60}")
    print(f"❓ Question: {question}")
    print(f"{'='*60}")
    
    # Step 1: Generate SQL
    print("\n⏳ Generating SQL...")
    sql = question_to_sql(question)
    print(f"\n🔍 SQL:\n{sql}\n")
    
    # Step 2: Run query
    result = run_query(sql)
    if result is None:
        return
    
    row_count = result.count()
    if row_count == 0:
        print(" No results found. Try rephrasing.")
        return
    
    # Step 3: Show results
    print(f"📊 Results ({row_count} rows):")
    display(result)
    
    # Step 4: Generate insight
    print("\n⏳ Generating insight...")
    insight = generate_insight(question, result)
    print(f"\n💡 Insight: {insight}")
    print(f"\n{'='*60}")

In [0]:
ask("Show high risk customers in Texas")

In [0]:
ask("What is the average default probability by loan grade?")

In [0]:
ask("Which states have the highest default rates?")

In [0]:
ask("Compare average interest rate between defaulted and fully paid loans")

In [0]:
# Create a text input widget at top of notebook
dbutils.widgets.removeAll()
dbutils.widgets.text("question", "", "Type your question here")

In [0]:
# Read from widget and run chatbot
question = dbutils.widgets.get("question")

if question.strip() == "":
    print("👆 Type a question in the box above and click Run")
else:
    ask(question)